In [1]:
import chromadb
import os
from dotenv import load_dotenv
from chromadb.utils import embedding_functions
from edgar import *
import yfinance as yf
from bs4 import BeautifulSoup
import requests
import re
import time

Connect to the Chroma Vector DB

In [2]:
# Path to .env location containing the Chroma API keys
env_path = os.path.join("../data", ".env")

# Parse the .env file and retrieve the API keys
if load_dotenv(env_path):
    print("✅ Environment variables loaded from data/.env")
else:
    print("❌ Failed to load data/.env - Check if the file exists!")

CF_CLIENT_ID = os.getenv("CF-ACCESS-CLIENT-ID")
CF_CLIENT_SECRET = os.getenv("CF-ACCESS-CLIENT-SECRET")
CHROMA_URL = os.getenv("CHROMA_URL")

print(f"ID: {CF_CLIENT_ID}")
print(f"Secret: {CF_CLIENT_SECRET}")
print(f"Chroma URL: {CHROMA_URL}")


✅ Environment variables loaded from data/.env
ID: 15703ac0e5797dac885c851f20afe6cb.access
Secret: 9db6c201a502a20d0aa9eae4dac06d36b2f6a29b46201c20df2f5174263ddf57
Chroma URL: chroma.taskcomply.com


In [3]:
# 2. Setup the Client with Custom Headers
client = chromadb.HttpClient(
    host=CHROMA_URL,                              # Your Cloudflare URL
    port=443,                                     # Standard HTTPS port
    ssl=True,                                     # MUST be True for Cloudflare
    headers={
        "CF-Access-Client-Id": CF_CLIENT_ID,
        "CF-Access-Client-Secret": CF_CLIENT_SECRET
    },
)

# 3. Test the connection
print(f"Connection Heartbeat: {client.heartbeat()}")

Connection Heartbeat: 1775457823131773864


Determine embedding function and create 2 streams:
- Fundamentals
- Sentiment

In [4]:
# 4. Use a lightweight embedding function
# Default is 'all-MiniLM-L6-v2' which is great for financial headlines
emb_fn = embedding_functions.DefaultEmbeddingFunction()

# 3. Create the two streams
fundamentals_col = client.get_or_create_collection(
    name="stream1_fundamentals",
    embedding_function=emb_fn
)

sentiment_col = client.get_or_create_collection(
    name="stream2_sentiment",
    embedding_function=emb_fn
)

In [5]:
collections = client.list_collections()

print(collections)

[Collection(name=stream1_fundamentals), Collection(name=stream2_sentiment)]


1. Stream 1: Ingesting Fundamentals (The "Ground Truth")
We will use edgartools to pull the latest 10-Ks. For RAG to be effective, we don't just want the whole document; we want the Risk Factors (Item 1A), as these provide the best "analytical anchor."

In [6]:
# Set tickers to ingest
tickers = ["NVDA", "AAPL", "TSLA"]

# MUST set your identity for SEC compliance
set_identity("david.j.bain@student.uts.edu.au")

In [7]:
company = Company("NVDA")
# Get the latest Annual Report
filing = company.get_filings(form="10-K").latest()

# Extract the specific "Risk Factors" section as Markdown
# EdgarTools 2026 handles the section parsing automatically
tenk = filing.obj()

risk = tenk.risk_factors

print("Fundamentals Stream Hydrated.")

Fundamentals Stream Hydrated.


2. Stream 2: Ingesting Sentiment (The "Market Pulse")
For the sentiment stream, we want the most recent headlines. Since Tesla (TSLA) just had a delivery miss on April 2nd, and Apple (AAPL) just celebrated its 50th anniversary on April 1st, there is plenty of fresh data.

In [6]:
print(collections[1])

Collection(name=stream2_sentiment)


In [27]:
# UPDATED
def get_verified_article_data(url, target_ticker, company_name):
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')

        # 1. CLEAN THE ENTIRE SOUP FIRST
        # Remove scripts, styles, and navigation so they don't mess up our word count
        for noise in soup.find_all(["script", "style", "nav", "footer", "aside", "header"]):
            noise.decompose()

        # 2. FULL BODY TEXT SCAN
        # soup.get_text() now only sees the "Visible" text of the article
        entire_page_text = soup.get_text(separator=" ").upper()

        # 3. DENSITY CHECK (The "Significance" Rule)
        ticker_count = entire_page_text.count(target_ticker.upper())
        name_count = entire_page_text.count(company_name.upper())
        total_mentions = ticker_count + name_count

        # 4. THE DECISION ENGINE
        # Tier 1: Is it in the Title? (Automatic Pass)
        page_title = soup.find('title').get_text().upper() if soup.find('title') else ""

        is_relevant = False
        if target_ticker.upper() in page_title or company_name.upper() in page_title:
            is_relevant = True
        # Tier 2: Is it mentioned at least 3 times throughout the whole article?
        elif total_mentions >= 3:
            is_relevant = True

        if not is_relevant:
            return None

        # 5. EXTRACTION (Find the actual article container)
        # We find the div with the highest density of <p> tags
        best_container = None
        max_p = 0
        for div in soup.find_all(['div', 'article', 'main', 'section']):
            p_count = len(div.find_all('p', recursive=False))
            if p_count > max_p:
                max_p = p_count
                best_container = div

        if best_container:
            # Capture headers and paragraphs for the final output
            elements = best_container.find_all(['p', 'h1', 'h2', 'h3'])
            full_text = "\n\n".join([e.get_text().strip() for e in elements if len(e.get_text()) > 20])
            return full_text if len(full_text) > 400 else None

        return None
    except Exception:
        return None

In [28]:
def inspect_discovered_news(ticker_sym, company_name):
    """
    Discovery and Verification ONLY. No writing to Chroma.
    """
    search_queries = [ticker_sym, company_name, f"{company_name} stock"]
    discovered_pool = {}

    # 1. Discovery Phase
    for q in search_queries:
        search_results = yf.Search(q, news_count=20).news
        for item in search_results:
            discovered_pool[item['uuid']] = item

    print(f"🔎 Total unique articles discovered: {len(discovered_pool)}")
    print("-" * 50)

    # 2. Verification & Staging Phase
    verified_articles = []

    for uuid, item in discovered_pool.items():
        # Use our existing 'Header-Aware' scraping function
        # This returns None if it's not the primary ticker

        print(f"UUID: {uuid}\n")
        print(f"Item: {item['link']}\n")

        content = get_verified_article_data(item['link'], ticker_sym, company_name)

        if content:
            # We store it in a dictionary for easy inspection
            verified_articles.append({
                "headline": item['title'],
                "body_snippet": content[:300] + "...", # Just a preview
                "full_length": len(content),
                "related": item.get('relatedTickers', []),
                "url": item['link']
            })
            print(f"✅ Verified: {item['title'][:60]}")
        else:
            print(f"❌ Rejected: {item['title'][:60]}")

        time.sleep(0.3) # Respect 2026 rate limits

    return verified_articles, discovered_pool, content

# --- RUN THE INSPECTION ---
target = "NVDA"
name = "Nvidia"
staged_data, discovered, content = inspect_discovered_news(target, name)

# --- VIEW THE RESULTS ---
print(f"\n--- INSPECTION REPORT: {len(staged_data)} STAGED ARTICLES ---")
for i, art in enumerate(staged_data):
    print(f"\n[{i+1}] {art['headline']}")
    print(f"    Related: {art['related']}")
    print(f"    Snippet: {art['body_snippet']}")

🔎 Total unique articles discovered: 22
--------------------------------------------------
UUID: 05b391fe-15fc-3302-a3c1-e757f09855b9

Item: https://finance.yahoo.com/markets/stocks/articles/nvidia-nvda-narrative-shifting-us-060508868.html

✅ Verified: How The Nvidia (NVDA) Narrative Is Shifting With US$1t AI Or
UUID: 283b19e1-eb8a-372a-9c77-edb0fb563275

Item: https://finance.yahoo.com/m/283b19e1-eb8a-372a-9c77-edb0fb563275/nvidia-ceo-jensen-huang-says.html

✅ Verified: NVIDIA CEO Jensen Huang Says AGI Is Here. If He’s Right, The
UUID: f7027b0b-6b14-3517-8852-7a7547f47cb5

Item: https://finance.yahoo.com/m/f7027b0b-6b14-3517-8852-7a7547f47cb5/cathie-wood-buys-%246.9-million.html

✅ Verified: Cathie Wood buys $6.9 million of surging tech stock
UUID: 8306bf89-ff56-367c-adee-79c8648db6b3

Item: https://finance.yahoo.com/markets/stocks/articles/nvidia-trillion-dollar-ai-goal-081310903.html

✅ Verified: Nvidia’s Trillion Dollar AI Goal And Groq Deal Reframe Valua
UUID: b00c0e5b-8bee-3e81-ba

In [26]:
def get_bulk_verified_news_and_write_to_chromadb(ticker_sym, company_name, collection):
    """
    Runs multiple searches to maximize article discovery.
    """
    # 1. Expand the Net
    # We search the ticker, the name, and a specific sector phrase
    search_queries = [
        ticker_sym,
        company_name,
        f"{company_name} stock sentiment"
    ]

    discovered_news = {}
    for q in search_queries:
        # We request 20, though Yahoo often soft-caps at 10-15
        search_results = yf.Search(q, news_count=20).news
        for item in search_results:
            discovered_news[item['uuid']] = item

    print(f"🔎 Total unique articles found for {ticker_sym}: {len(discovered_news)}")

    # 2. Filter and Scrape
    for uuid, item in discovered_news.items():
        # Skip if already in ChromaDB
        if collection.get(ids=[uuid])['ids']:
            continue

        # Use our 'Header-Aware' and 'Primary Ticker' scraping function
        # This ensures that even if 'Nvidia' was found via a broad search,
        # it's only saved if it's the primary subject.
        content = get_verified_article_data(item['link'], ticker_sym)

        if content:
            collection.add(
                documents=[content],
                metadatas=[{
                    "ticker": ticker_sym,
                    "headline": item['title'],
                    "related": ", ".join(item.get('relatedTickers', [])),
                    "source": item.get('publisher', 'Unknown')
                }],
                ids=[uuid]
            )
            print(f"✅ Verified & Saved: {item['title'][:55]}...")

        # Brief pause to respect the 2026 rate limits
        time.sleep(0.4)

In [27]:
'''
# The "Discovery" List
stocks_to_track = [
    {"ticker": "NVDA", "name": "Nvidia"},
    {"ticker": "TSLA", "name": "Tesla"},
    {"ticker": "AAPL", "name": "Apple"}
]

for stock in stocks_to_track:
    print(f"\n--- Harvesting {stock['ticker']} ---")
    get_bulk_verified_news_and_write_to_chromadb(stock['ticker'], stock['name'], sentiment_col)
'''


--- Processing NVDA ---
🔎 Found 10 unique articles across all search variations.

--- Processing AAPL ---
🔎 Found 10 unique articles across all search variations.

--- Processing TSLA ---
🔎 Found 11 unique articles across all search variations.
